In [2]:
import pandas as pd
import calendar

path = "../data/raw/DADS5250_Hackathon_Google_Data.xlsx"

forecast = pd.read_csv("../data/processed/forecast.csv")
params = pd.read_excel(path, sheet_name="Agent Salary and Threshold")
staff = pd.read_excel(path, sheet_name="Existing Agent Staffing").set_index("Country")["Existing_Agent_Count"]

print(forecast.shape, params.shape, staff.shape)

(264, 6) (22, 3) (22,)


In [3]:
import calendar

SUPPORT_DAYS = 60     # an advertiser is supported for 60 days
MAX_WAIT_DAYS = 60    # leaves the pool after waiting 60 days
CAPACITY = 10         # advertisers per agent at once
UPLIFT = 0.135        # expected uplift: 13.5% of annual budget (assumption A1)
FIRE_COST = 0.40      # firing costs 40% of annual salary

salaries = params.set_index("Country")["Annual_Agent_Salary_USD"]

In [4]:
def run_plan(country, headcount, scenario="base"):
    salary = salaries[country]
    fc = forecast[forecast["country"] == country].set_index("month")
    budget = fc["avg_eligible_budget"].iloc[0]
    start_agents = staff[country]

    pool = []      # people waiting:  [arrival_day, how_many]
    active = []    # people served:   [start_day, how_many]
    rows = []
    day = 0

    for month in range(1, 13):
        agents = headcount[month - 1]
        arrivals_per_day = fc.loc[month, f"daily_eligible_{scenario}"]
        n_days = calendar.monthrange(2026, month)[1]
        served = abandoned = wait_days = busy = 0.0

        for _ in range(n_days):
            # 1. graduate: keep only groups that started fewer than 60 days ago
            active = [g for g in active if day - g[0] < SUPPORT_DAYS]

            # 2. abandon: anyone who has waited 60 days leaves
            still_waiting = []
            for g in pool:
                if day - g[0] >= MAX_WAIT_DAYS:
                    abandoned += g[1]
                else:
                    still_waiting.append(g)
            pool = still_waiting

            # 3. today's arrivals join the pool
            pool.append([day, arrivals_per_day])

            # 4. fill free slots, oldest first
            free = agents * CAPACITY - sum(g[1] for g in active)
            while free > 1e-9 and pool:
                g = pool[0]
                take = min(free, g[1])
                active.append([day, take])
                served += take
                wait_days += take * (day - g[0])
                g[1] -= take
                free -= take
                if g[1] <= 1e-9:
                    pool.pop(0)

            busy += sum(g[1] for g in active)
            day += 1

        # hiring and firing this month
        prev = start_agents if month == 1 else headcount[month - 2]
        hires = max(agents - prev, 0)
        fires = max(prev - agents, 0)
        cost = (agents * salary / 12           # salaries this month
                + hires * salary / 12          # new hires paid during their ramp-up month
                + fires * salary / 12          # fired agents paid during their notice month
                + fires * FIRE_COST * salary)  # firing penalty

        rows.append({
            "month": month, "agents": agents, "hires": hires, "fires": fires,
            "served": served, "abandoned": abandoned,
            "avg_wait_days": wait_days / served if served else 0,
            "utilization": busy / (agents * CAPACITY * n_days) if agents else 0,
            "waiting_end": sum(g[1] for g in pool),
            "revenue": served * budget * UPLIFT,
            "cost": cost,
        })

    return pd.DataFrame(rows)

In [5]:
plans = {
    "Keep all 652":       [652] * 12,
    "Peak need 296":      [296] * 12,
    "Too few: 200":       [200] * 12,
}

results = []
for name, plan in plans.items():
    m = run_plan("USA", plan)
    results.append({
        "plan": name,
        "revenue_M": round(m["revenue"].sum() / 1e6, 1),
        "cost_M": round(m["cost"].sum() / 1e6, 1),
        "net_M": round((m["revenue"].sum() - m["cost"].sum()) / 1e6, 1),
        "served": round(m["served"].sum()),
        "abandoned": round(m["abandoned"].sum()),
        "max_wait_days": round(m["avg_wait_days"].max(), 1),
    })
pd.DataFrame(results)

,plan,revenue_M,cost_M,net_M,served,abandoned,max_wait_days
0,Keep all 652,151.3,50.7,100.7,13043,0,0.0
1,Peak need 296,151.3,36.4,115.0,13043,0,0.0
2,Too few: 200,135.8,32.5,103.3,11705,0,24.3


In [6]:
  run_plan("USA", [200] * 12).round(1)

,month,agents,hires,fires,served,abandoned,avg_wait_days,utilization,waiting_end,revenue,cost
0,1,200,0,452,1024.0,0.0,0.0,0.3,0.0,11881580.2,18274797.8
1,2,200,0,0,976.0,0.0,0.0,0.8,4.1,11324631.1,1295350.0
2,3,200,0,0,991.0,0.0,1.1,1.0,36.6,11498303.4,1295350.0
3,4,200,0,0,1008.6,0.0,0.3,1.0,0.0,11702906.6,1295350.0
4,5,200,0,0,943.0,0.0,0.0,1.0,0.0,10941728.6,1295350.0
5,6,200,0,0,858.5,0.0,0.0,0.9,0.0,9961266.2,1295350.0
6,7,200,0,0,914.5,0.0,0.0,0.9,0.0,10611040.1,1295350.0
7,8,200,0,0,919.0,0.0,0.0,0.9,0.0,10663254.1,1295350.0
8,9,200,0,0,1063.0,0.0,0.0,0.9,0.0,12334101.3,1295350.0
9,10,200,0,0,972.4,0.0,4.8,1.0,526.1,11283246.7,1295350.0


In [7]:
run_plan("Vietnam", [59] * 12).round(1)

,month,agents,hires,fires,served,abandoned,avg_wait_days,utilization,waiting_end,revenue,cost
0,1,59,0,0,382.5,0.0,0.0,0.3,0.0,893459.4,88028.0
1,2,59,0,0,207.5,0.0,0.0,0.9,164.4,484687.1,88028.0
2,3,59,0,0,370.2,0.0,14.2,1.0,181.3,864638.1,88028.0
3,4,59,0,0,219.8,0.0,14.1,1.0,338.9,513508.4,88028.0
4,5,59,0,0,382.5,0.0,27.2,1.0,319.4,893459.4,88028.0
5,6,59,0,0,219.8,0.0,26.9,1.0,439.6,513508.4,88028.0
6,7,59,0,0,383.4,0.0,37.4,1.0,408.2,895667.1,88028.0
7,8,59,0,0,231.2,0.0,36.6,1.0,482.4,540121.9,88028.0
8,9,59,0,0,372.1,0.0,44.6,1.0,501.9,869053.6,88028.0
9,10,59,0,0,230.3,0.0,41.9,1.0,830.6,537914.2,88028.0
